In [ ]:
# ============================================================================
# CELL 1 — Load anonymised JSON files and create the schema
# ============================================================================
# Reads the anonymised files written by the previous notebook directly from
# the volume. No dependency on in-memory state from the previous notebook.

import json
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, LongType,
    DoubleType, TimestampType, BooleanType,
)
from datetime import datetime, timedelta

CATALOG = "multisport_training"
SCHEMA  = "training"
FQ_SCHEMA = f"{CATALOG}.{SCHEMA}"

ANON_DIR = f"/Volumes/{CATALOG}/default/training_data_anon"


def list_json_files(root):
    out = []
    stack = [root]
    while stack:
        current = stack.pop()
        try:
            entries = dbutils.fs.ls(current)
        except Exception as e:
            print(f"WARN: cannot list {current}: {e}")
            continue
        for e in entries:
            if e.isDir():
                stack.append(e.path)
            elif e.name.endswith(".json"):
                out.append(e.path)
    return out


def read_text(path):
    return dbutils.fs.head(path, 64 * 1024 * 1024)


# Load every anonymised file
files = list_json_files(ANON_DIR)
print(f"Found {len(files)} anonymised JSON files under {ANON_DIR}")

anon_sessions = []
failed = 0
for fp in files:
    try:
        anon_sessions.append(json.loads(read_text(fp)))
    except Exception as e:
        print(f"WARN: cannot read {fp}: {e}")
        failed += 1

print(f"Loaded {len(anon_sessions)} sessions (failed {failed})")

# Create the schema
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {FQ_SCHEMA}")
print(f"Schema ready: {FQ_SCHEMA}")

tables = [r.tableName for r in spark.sql(f"SHOW TABLES IN {FQ_SCHEMA}").collect()]
 
for t in sorted(tables):
    fq = f"{FQ_SCHEMA}.{t}"
    print(f"\n=== {fq} ===")
    display(spark.sql(f"SELECT * FROM {fq} LIMIT 2"))

In [ ]:
# ============================================================================
# CELL 2 — Build `users` and `sessions` tables
# ============================================================================
# Both are Delta tables, overwritten on each run. Sessions has one row per
# (session, exercise) -- so a triathlon with 3 legs produces 3 rows here.

from datetime import datetime, timedelta

# ---- USERS ----------------------------------------------------------------
# One row per user. Just identity + activity span. No thresholds.
users_data = {}  # user_id -> dict
for s in anon_sessions:
    uid = s["user_id"]
    start = s.get("start_time_iso")
    if uid not in users_data:
        users_data[uid] = {
            "user_id": uid,
            "first_session_time": start,
            "last_session_time": start,
            "session_count": 0,
        }
    rec = users_data[uid]
    rec["session_count"] += 1
    if start and (rec["first_session_time"] is None or start < rec["first_session_time"]):
        rec["first_session_time"] = start
    if start and (rec["last_session_time"] is None or start > rec["last_session_time"]):
        rec["last_session_time"] = start

users_rows = list(users_data.values())
users_schema = StructType([
    StructField("user_id",            StringType(),    False),
    StructField("first_session_time", TimestampType(), True),
    StructField("last_session_time",  TimestampType(), True),
    StructField("session_count",      IntegerType(),   False),
])
# Convert ISO strings to datetime for the timestamp columns
for r in users_rows:
    r["first_session_time"] = datetime.fromisoformat(r["first_session_time"]) if r["first_session_time"] else None
    r["last_session_time"]  = datetime.fromisoformat(r["last_session_time"])  if r["last_session_time"]  else None

(spark.createDataFrame(users_rows, users_schema)
      .write.format("delta").mode("overwrite")
      .saveAsTable(f"{FQ_SCHEMA}.users"))
print(f"Wrote {FQ_SCHEMA}.users -- {len(users_rows)} rows")


# ---- SESSIONS -------------------------------------------------------------
# One row per (session, exercise). Triathlon legs = separate rows here.
sessions_rows = []
for s in anon_sessions:
    th = s.get("athlete_thresholds", {}) or {}
    for ex in s["exercises"]:
        # Compute end_time and duration_s from samples length
        streams = ex["samples"]["streams"]
        interval_ms = ex["samples"]["interval_ms"]
        # Use the longest stream as the canonical length
        n = max(len(v) for v in streams.values()) if streams else 0
        duration_s = (n * interval_ms) / 1000.0 if interval_ms else None
        try:
            start_dt = datetime.fromisoformat(ex["start_time_iso"])
            end_dt = start_dt + timedelta(seconds=duration_s) if duration_s is not None else None
        except (ValueError, TypeError):
            start_dt = None
            end_dt = None

        sessions_rows.append({
            "user_id":                       s["user_id"],
            "session_id":                    s["session_id"],
            "exercise_id":                   ex["exercise_id"],
            "exercise_index":                ex["exercise_index"],
            "session_sport_id":              str(s.get("sport_id")) if s.get("sport_id") is not None else None,
            "session_sport_name":            s.get("sport_name"),
            "exercise_sport_id":             str(ex.get("sport_id")) if ex.get("sport_id") is not None else None,
            "exercise_sport_name":           ex.get("sport_name"),
            "start_time":                    start_dt,
            "end_time":                      end_dt,
            "duration_s":                    duration_s,
            "tz_offset_minutes":             s.get("timezone_offset_minutes"),
            "hr_max_at_session":             th.get("hr_max"),
            "hr_resting_at_session":         th.get("hr_resting"),
            "aerobic_threshold_at_session":  th.get("aerobic_threshold"),
            "anaerobic_threshold_at_session":th.get("anaerobic_threshold"),
            "vo2max_at_session":             th.get("vo2max"),
            "ftp_at_session":                th.get("ftp"),
            "mas_kmh_at_session":            th.get("mas_kmh"),
            "map_watts_at_session":          th.get("map_watts"),
        })

sessions_schema = StructType([
    StructField("user_id",                       StringType(),    False),
    StructField("session_id",                    StringType(),    False),
    StructField("exercise_id",                   StringType(),    False),
    StructField("exercise_index",                IntegerType(),   False),
    StructField("session_sport_id",              StringType(),    True),
    StructField("session_sport_name",            StringType(),    True),
    StructField("exercise_sport_id",             StringType(),    True),
    StructField("exercise_sport_name",           StringType(),    True),
    StructField("start_time",                    TimestampType(), True),
    StructField("end_time",                      TimestampType(), True),
    StructField("duration_s",                    DoubleType(),    True),
    StructField("tz_offset_minutes",             IntegerType(),   True),
    StructField("hr_max_at_session",             IntegerType(),   True),
    StructField("hr_resting_at_session",         IntegerType(),   True),
    StructField("aerobic_threshold_at_session",  IntegerType(),   True),
    StructField("anaerobic_threshold_at_session",IntegerType(),   True),
    StructField("vo2max_at_session",             DoubleType(),    True),
    StructField("ftp_at_session",                IntegerType(),   True),
    StructField("mas_kmh_at_session",            DoubleType(),    True),
    StructField("map_watts_at_session",          IntegerType(),   True),
])

(spark.createDataFrame(sessions_rows, sessions_schema)
      .write.format("delta").mode("overwrite")
      .saveAsTable(f"{FQ_SCHEMA}.sessions"))
print(f"Wrote {FQ_SCHEMA}.sessions -- {len(sessions_rows)} rows")

In [ ]:
# ============================================================================
# CELL 3 — Build one `samples_<sport>` Delta table per session-level sport
# ============================================================================
# Splits exercises by their session-level sport_name (so triathlon and
# multisport sessions stay together in one table). Each row is one second
# of one exercise. exercise_sport_name is preserved on each row so you can
# filter to specific legs of a triathlon.

from datetime import datetime, timedelta
import re

# Common schema for all samples tables. Sports without a given stream just
# get nulls in that column.
samples_schema = StructType([
    StructField("user_id",             StringType(),    False),
    StructField("session_id",          StringType(),    False),
    StructField("exercise_index",      IntegerType(),   False),
    StructField("second",              IntegerType(),   False),
    StructField("timestamp",           TimestampType(), True),
    StructField("exercise_sport_id",   StringType(),    True),
    StructField("exercise_sport_name", StringType(),    True),
    StructField("heart_rate",          DoubleType(),    True),
    StructField("speed_kmh",           DoubleType(),    True),
    StructField("cadence",             DoubleType(),    True),
    StructField("distance_m",          DoubleType(),    True),
    StructField("altitude_m",          DoubleType(),    True),
    StructField("temperature_c",       DoubleType(),    True),
    StructField("is_paused",           BooleanType(),   False),
])


def sport_to_table_name(sport_name: str) -> str:
    """Convert sport_name like 'POOL_SWIMMING' to a safe table name like
    'samples_pool_swimming'. Strips anything that isn't alphanumeric/underscore."""
    if not sport_name:
        return "samples_unknown_none"
    safe = re.sub(r"[^A-Za-z0-9_]+", "_", sport_name).strip("_").lower()
    return f"samples_{safe}"


def build_sample_rows(session, exercise):
    """Yield one dict per second of this exercise."""
    streams = exercise["samples"]["streams"]
    interval_ms = exercise["samples"]["interval_ms"] or 1000
    n = max((len(v) for v in streams.values()), default=0)
    if n == 0:
        return

    try:
        start_dt = datetime.fromisoformat(exercise["start_time_iso"])
    except (ValueError, TypeError):
        start_dt = None

    pauses = exercise.get("pause_intervals_ms") or []

    hr   = streams.get("heart_rate")
    spd  = streams.get("speed")
    cad  = streams.get("cadence")
    dst  = streams.get("distance")
    alt  = streams.get("altitude")
    tmp  = streams.get("temperature")

    user_id   = session["user_id"]
    sess_id   = session["session_id"]
    ex_idx    = exercise["exercise_index"]
    ex_sport_id   = str(exercise.get("sport_id")) if exercise.get("sport_id") is not None else None
    ex_sport_name = exercise.get("sport_name")

    for i in range(n):
        ms = i * interval_ms
        is_paused = any(ps <= ms < pe for ps, pe in pauses)
        ts = start_dt + timedelta(milliseconds=ms) if start_dt else None
        yield {
            "user_id":             user_id,
            "session_id":          sess_id,
            "exercise_index":      ex_idx,
            "second":              i,
            "timestamp":           ts,
            "exercise_sport_id":   ex_sport_id,
            "exercise_sport_name": ex_sport_name,
            "heart_rate":          hr[i]  if hr  and i < len(hr)  else None,
            "speed_kmh":           spd[i] if spd and i < len(spd) else None,
            "cadence":             cad[i] if cad and i < len(cad) else None,
            "distance_m":          dst[i] if dst and i < len(dst) else None,
            "altitude_m":          alt[i] if alt and i < len(alt) else None,
            "temperature_c":       tmp[i] if tmp and i < len(tmp) else None,
            "is_paused":           is_paused,
        }


# Group exercises by session-level sport_name (so triathlon stays whole)
buckets: dict[str, list] = {}
for s in anon_sessions:
    table_name = sport_to_table_name(s.get("sport_name"))
    bucket = buckets.setdefault(table_name, [])
    for ex in s["exercises"]:
        for row in build_sample_rows(s, ex):
            bucket.append(row)

# Write each bucket as its own Delta table
for table_name, rows in sorted(buckets.items()):
    if not rows:
        continue
    fq_table = f"{FQ_SCHEMA}.{table_name}"
    (spark.createDataFrame(rows, samples_schema)
          .write.format("delta").mode("overwrite")
          .saveAsTable(fq_table))
    print(f"Wrote {fq_table} -- {len(rows):,} rows")

print(f"\nDone. {len(buckets)} samples tables created.")

In [ ]:
# ============================================================================
# CELL 4 — Verify with SQL
# ============================================================================
# Lists all tables in the schema and runs a couple of sanity queries to
# confirm the joins work as expected.

print("=== Tables in schema ===")
display(spark.sql(f"SHOW TABLES IN {FQ_SCHEMA}"))

print("\n=== Users ===")
display(spark.sql(f"SELECT * FROM {FQ_SCHEMA}.users ORDER BY session_count DESC"))

print("\n=== Sessions per sport ===")
display(spark.sql(f"""
    SELECT session_sport_name,
           COUNT(*) AS n_exercises,
           COUNT(DISTINCT session_id) AS n_sessions,
           ROUND(SUM(duration_s) / 3600, 1) AS total_hours
    FROM {FQ_SCHEMA}.sessions
    GROUP BY session_sport_name
    ORDER BY n_exercises DESC
"""))

print("\n=== Sample row counts per sport table ===")
samples_tables = [r.tableName for r in spark.sql(f"SHOW TABLES IN {FQ_SCHEMA}").collect()
                  if r.tableName.startswith("samples_")]
for t in sorted(samples_tables):
    cnt = spark.sql(f"SELECT COUNT(*) AS n FROM {FQ_SCHEMA}.{t}").collect()[0]["n"]
    print(f"  {t}: {cnt:,} rows")

# Example: join sessions to a samples table
print("\n=== Example join: avg HR per running session ===")
display(spark.sql(f"""
    SELECT s.session_id,
           s.start_time,
           ROUND(s.duration_s / 60, 1) AS duration_min,
           ROUND(AVG(sm.heart_rate), 0) AS avg_hr,
           ROUND(MAX(sm.heart_rate), 0) AS max_hr,
           ROUND(MAX(sm.distance_m), 0) AS distance_m
    FROM {FQ_SCHEMA}.sessions s
    JOIN {FQ_SCHEMA}.samples_running sm
      ON s.session_id = sm.session_id
     AND s.exercise_index = sm.exercise_index
    WHERE s.session_sport_name = 'RUNNING'
      AND NOT sm.is_paused
    GROUP BY s.session_id, s.start_time, s.duration_s
    ORDER BY s.start_time DESC
    LIMIT 10
"""))

In [ ]:
display(spark.sql(f"SHOW TABLES IN {FQ_SCHEMA}"))